# Finetuning and Deploying Hugging Face Models on Vertex AI

## Overview

This tutorial will focus on utilizing Hugging Face which is a repository for user to share and download machine learning models, datasets, and demos. For this tutorial we will load in a model and dataset from Hugging Face and train and test our model before deploying it on Vertex AI. The model we will be deploying is Flan T5 and the datasets is [ccdv/pubmed-summarization](https://HuggingFace.co/datasets/ccdv/pubmed-summarization). Steps will show how to hypertune a model locally and how to launch our custom training job on Vertex AI Training, these steps are based on Keras NLP Tutorials for [abstractive summarization](https://keras.io/examples/nlp/t5_hf_summarization/).

You may be wondering why are we training a pretrained model? The reason for this is because we are fine tuning our pretrained model for optimal performance on a particular application, in our case summarizing scientific documents. This is not a necessary step anymore as new methods have been made to enhance model performance like zero-shot learning which we will go over in our next tutorial.

## Prerequisites 
For this tutorial it is recommended to use 1 GPU to speed up processes, this notebooks was run using the machinetype n1-highcpu-8 (8 vCPUs, 7.199 GB RAM) on Tensorflow. Visit the following tutorial to set up notebooks that utilize: GPUs [Spinning up a Vertex AI Notebook](/docs/vertexai.md).

## Learning Objectives
+ Learning the basics of Hugging Face models
+ Learn how to deploy a Hugging Face model and fine tune it

## Get Started

### Install packages

Hugging Face **transformers** are an open-source framework that allows you to utilize APIs and tools to download pretrained models, set hyperparameters, tokenize datasets, and further tune them to suite your needs. Here we are updating Vertex AI as well as installing the transformers package and **datasets** so that we can have access to Hugging Face datasets and as a bonus we are adding the S3 feature to help download datasets that may already be in a S3 bucket.

In [ ]:
!pip install "transformers" "datasets" "rouge_score" "evaluate" 
!pip install --upgrade --force-reinstall --no-cache-dir scikit-learn
!pip install --upgrade --no-cache-dir numpy transformers datasets accelerate torch
!pip install --no-cache-dir --force-reinstall torch torchvision torchaudio

## Get Started

### Hardware Acceleration and Global Configuration

Configures the computational device (routes math to GPU if available; otherwise defaults to CPU) and defines the pre-trained architecture baseline.

**google-t5/t5-small** is an open-source, lightweight encoder-decoder Transformer model developed by Google with 60 million parameters. Built on a unified "text-to-text" framework, it converts all NLP tasks—including translation, summarization, and classification—into a text generation format. Because it was pre-trained on a massive web text corpus (C4) but retains a compact size, it is highly efficient, runs quickly on standard hardware, and serves as an ideal baseline for prototyping or fine-tuning specialized tasks on a budget.

In [ ]:
import os
import torch

# Global model checkpoint from Hugging Face Hub
CHECKPOINT = "google-t5/t5-small"

# Verify hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using execution device: {device}")


### Loading the Source Dataset

Downloads the SAMSum messenger dialogue corpus and splits it into structured train and test sets.

In [ ]:
from datasets import load_dataset

# Load a classic text/summarization dataset
raw_datasets = load_dataset("knkarthick/samsum", trust_remote_code=True)

# Split dataset components into standard train and test variables
train = raw_datasets["train"]
test = raw_datasets["test"]

print("Dataset Loaded Successfully!")
print("Train samples: "+len(train))
print("Test samples: "+len(test))
print("Sample item layout:", train[0].keys())

### Text Tokenization and Mapping Preprocessing

 Converts human text strings into uniform numerical arrays (token integer ID vectors) that the transformer can process.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def preprocess_function(examples):
    # Adjust 'dialogue' and 'summary' to match your explicit dataset key names
    inputs = [doc for doc in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Tokenize the targets/labels
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Run mapping preprocessing over the datasets
tokenized_train = train.map(preprocess_function, batched=True)
tokenized_test = test.map(preprocess_function, batched=True)

### Constructing Compliant PyTorch DataLoaders

Dynamically pads data, prunes away unneeded text string columns, and aggregates the numerical representations into iterative chunks (batches).

In [ ]:
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader

# Create a data collator to handle sequence dynamic padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=None, return_tensors="pt")

# Dynamically filter out columns that are raw text strings
columns_to_remove = [col for col in tokenized_train.column_names if col not in ["input_ids", "attention_mask", "labels"]]
print(f"Filtering out raw string meta columns: {columns_to_remove}")

train_set = tokenized_train.remove_columns(columns_to_remove)
test_set = tokenized_test.remove_columns(columns_to_remove)

# Build compliant PyTorch DataLoaders
train_dataloader = DataLoader(
    train_set, 
    shuffle=True, 
    batch_size=8, 
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    test_set, 
    batch_size=8, 
    collate_fn=data_collator
)

print("DataLoaders assembled successfully")

### Initializing the Model and Optimization Engine

Loads the pre-trained weights from Hugging Face onto your execution hardware and prepares the mathematical weight optimization parameters.

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from torch.optim import AdamW

# Initialize via PyTorch wrapper. 'from_tf=True' automatically maps old weights if needed
model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT, from_tf=False)
model.to(device)

# Initialize standard PyTorch native optimizer (Replacing deprecated transformers.AdamW)
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f"Model loaded and pushed to {device}")

### Local Model Training Execution Loop

Runs the custom fine-tuning optimization phase, calculating losses via backpropagation and iteratively updating the internal network paths.

We will use the validation dataset for calculating our ROUGE score. While our ROUGE score is being calculated and our training is running its best to set up a **callback system**. A callback is an object that can perform actions at various stages of training and helps to write logs after every batch of training to monitor your metrics, periodically save your model to disk, and if need be do early stopping. Here we are using Keras call back system.

In [ ]:
num_epochs = 1
model.train()

print(" Starting model training loop...")
for epoch in range(num_epochs):
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        # Ship all numeric input feature maps directly to your GPU/CPU device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Compute forward pass predictions
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Backpropagation pass
        loss.backward()
        optimizer.step()
        
        if step % 10 == 0:
            print(f"Epoch: {epoch+1} | Step: {step}/{len(train_dataloader)} | Running Loss: {loss.item():.4f}")

print("\n Training Loop completed successfully")


### Metric Suite Preparation

 Loads the quantitative ROUGE scorer metric used to score textual overlap against ground truth labels.

In [ ]:
!pip install --upgrade --no-cache-dir evaluate rouge_score

import evaluate

# Load the classic ROUGE metric used for validation evaluation
metric = evaluate.load("rouge")
print("Metric suite loaded and ready")

### Testing the Model

Here we will use a sample text that we want our model to summarize.

In [ ]:
text = "Severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) is a \
highly transmissible and pathogenic coronavirus that emerged in late 2019 and has \
caused a pandemic of acute respiratory disease, named ‘coronavirus disease 2019’ (COVID-19), \
which threatens human health and public safety. In this Review, we describe the basic virology of \
SARS-CoV-2, including genomic characteristics and receptor use, highlighting its key difference \
from previously known coronaviruses. We summarize current knowledge of clinical, epidemiological and \
pathological features of COVID-19, as well as recent progress in animal models and antiviral treatment \
approaches for SARS-CoV-2 infection. We also discuss the potential wildlife hosts and zoonotic origin \
of this emerging virus in detail."

xt summarization inference by processing input text, generating a response, and decoding the output. It starts by placing the model into evaluation mode via model.eval(), which ensures consistent outputs by disabling training-specific layers like dropout, while with torch.no_grad() speeds up execution and saves memory by skipping gradient calculations. The core behavior is governed by the critical parameters inside the model.generate() function, which control how the text is written:
- **inputs["input_ids"] and attention_mask:**  Pass the numerical text tokens and a mask indicating which tokens are actual text versus padding, guiding the model on what to focus on.
- **max_length=64:** Enforces a strict upper bound on the output length, ensuring the generated summary does not exceed 64 tokens.
- **num_beams=4:**  Activates Beam Search with a width of 4, meaning the model evaluates the 4 most probable word sequences at each step rather than greedily picking the single next word, which significantly boosts text quality.
- **early_stopping=True:** Tells the generation loop to stop as soon as all 4 beams reach a definitive end-of-sentence token, saving computation time.

To predict the following tokenizes the text to gather the inputs, then uses **generate()** generate sequences of token ids for our model. We then decode our output to translate our tokenized output into text.

In [ ]:
# 1. Put the model into evaluation mode (disables dropout layers, etc.)
model.eval()

inputs = tokenizer(text, max_length=512, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    output_tokens = model.generate(
        inputs["input_ids"], 
        attention_mask=inputs["attention_mask"],
        max_length=64, 
        num_beams=4, 
        early_stopping=True
    )

summary_output = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(" Inferred Summary Output:")
print("-" * 50)
print(summary_output)
print("-" * 50)

### Optional: Summarizing PDF Files

The process of summarizing scientific PDF files is relatively the same except that we first need to extract the text from the PDF. To do so lets download a PDF file from PubMed.

In [ ]:
! wget --user-agent="Chrome" https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7784226/pdf/12248_2020_Article_532.pdf

We'll be downloading some tools that help us extract only the text from our pdf file.

In [ ]:
!pip install pymupdf

Now we can make a function **extract_text_from_pdf** to extract the text from the pdf and save it as a variable.

In [ ]:
import fitz

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ''
    for page in doc:
        text += page.get_text()
    return text

text_pdf=extract_text_from_pdf('12248_2020_Article_532.pdf')

Finally we'll follow the same steps we did before to encode our inputs, pass it to our model, and then decode our output. Notice how we increased the max_length of what is expected of our input.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
inputs = tokenizer( text_pdf, max_length=1000, truncation=True, return_tensors="pt")
model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# 5. Generate outputs safely using the dictionary unpacked inputs
with torch.no_grad():
    outputs = model.generate(
        **inputs,                  # Passes both input_ids and attention_mask
        max_new_tokens=100, 
        do_sample=False,           # Uses stable greedy decoding
        early_stopping=True        # Prevents infinite repetitive loops
    )

# 6. Decode and display your inferred summary output text
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Output Summary:\n", summary)


### Finetuning our Model via Vertex AI Training API

### Setting up our Datasets for Training 

Although we have our datasets saved locally inorder to utilize the Vertex AI Training API we will need to store our datasets in a bucket.

In [ ]:
from datasets import load_dataset

# load dataset
train, test, validation = load_dataset("ccdv/pubmed-summarization", split=["train[:5%]", "test[:5%]", "validation[:5%]" ])

In [ ]:
#load in the storage package and name our bucket
from google.cloud import storage
BUCKET='flan-t5-model-resources'
client = storage.Client()

In [ ]:
#Create bucket
bucket = client.bucket(BUCKET)
bucket.create()

Convert our datasets to csv and upload to our bucket in one step!

In [ ]:
from io import BytesIO

#convert train dataset to csv and push to GCS bucket
csv_buffer = BytesIO()
train.to_csv(csv_buffer)
client = storage.Client()
bucket = client.get_bucket(BUCKET)
bucket.blob('train.csv').upload_from_file(csv_buffer, 'text/csv')

In [ ]:
#convert test dataset to csv and push to GCS bucket
csv_buffer = BytesIO()
test.to_csv(csv_buffer)
client = storage.Client()
bucket = client.get_bucket(BUCKET)
bucket.blob('test.csv').upload_from_file(csv_buffer, 'text/csv')

In [ ]:
#convert validation dataset to csv and push to GCS bucket
csv_buffer = BytesIO()
validation.to_csv(csv_buffer)
client = storage.Client()
bucket = client.get_bucket(BUCKET)
bucket.blob('validation.csv').upload_from_file(csv_buffer, 'text/csv')

Here we will be saving the location of our datasets be used when we execute the training of our model.

In [ ]:
# save train_dataset to s3
training_input_path = f'gs://{BUCKET}/train.csv'

# save test_dataset to s3
test_input_path = f'gs://{BUCKET}/test.csv'

validation_input_path = f'gs://{BUCKET}/validation.csv'

### Training our Model via Vertex AI Training API

To train our model on Vertex AI Training API you must first create a custom AI job, this is done by creating a autopkg that holds your requirements.txt and task.py files is a specific structure like so: 

```
autopkg-summarizer /
    + requirements.txt
    + trainer/
        + task.py
```

In [ ]:
#Creates the following directories and files
!mkdir autopkg-summarizer
!touch autopkg-summarizer/requirements.txt
!mkdir autopkg-summarizer/trainer
!touch autopkg-summarizer/trainer/task.py

Add your requirements.txt file by adding the packages below:
```
nltk
transformers
keras_nlp
datasets
rouge_score
torch
datasets 
accelerate 
evaluate
```

To create our training script we will be adding all the steps that we ran from the 'Finetuning our Model Locally' section of this tutorial to a file named task.py:

```
import nltk
import argparse
from datasets import load_dataset
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    DataCollatorForSeq2Seq, 
    set_seed,
    get_scheduler
)
import evaluate
from tqdm.auto import tqdm

def get_args():
    '''Parses args.'''
    parser = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument(
        '--model_name_or_path',
        required=True,
        type=str,
        help='name of model or path to load into tokenizer and class')
    parser.add_argument(
        '--train_file',
        required=True,
        type=str,
        help='train dataset in csv or json format')
    parser.add_argument(
        '--test_file',
        required=True,
        type=str,
        help='test dataset in csv or json format')
    parser.add_argument(
        '--validation_file',
        required=True,
        type=str,
        help='validation dataset in csv or json format used to calculate ROUGE score')
    parser.add_argument(
        '--text_column',
        required=True,
        type=str,
        help='The name of the column in the datasets containing the full texts (for summarization)')
    parser.add_argument(
        '--summary_column',
        required=True,
        type=str,
        help='The name of the column in the datasets containing the abstracts or summary of the full text')
    parser.add_argument(
        '--num_train_epochs',
        required=False,
        type=int,
        default=3,
        help='number of complete passes through the training dataset')
    parser.add_argument(
        '--source_prefix',
        required=False,
        type=str,
        help='A prefix to add before every source text (needed for T5 models)')
    parser.add_argument(
        '--inputs_max_length',
        required=False,
        type=int,
        default=1024,
        help='max token length for model inputs')
    parser.add_argument(
        '--labels_max_length',
        required=False,
        type=int,
        default=128,
        help='max token length for model labels or targets')
    parser.add_argument(
        '--batch_size',
        required=False,
        type=int,
        default=10,
        help='max token length for model labels or targets')
    parser.add_argument(
        '--output_dir',
        required=True,
        type=str,
        help='directory to store saved model')
    
    args = parser.parse_args()
    return args

def main():
    set_seed(42)
    args = get_args() 
 
    checkpoint = args.model_name_or_path
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    
    text = args.text_column
    summary = args.summary_column
    inputs_max_length = args.inputs_max_length
    labels_max_length = args.labels_max_length
    prefix = args.source_prefix 
    
    # Load PyTorch variant model
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint) 
    model.to(device)
    
    data_files = {'train': args.train_file, 'test': args.test_file, 'validation': args.validation_file}
    extension = args.train_file.split(".")[-1]
    
    raw_datasets = load_dataset(extension, data_files=data_files)
    raw_datasets = raw_datasets.filter(lambda x: x[text] is not None) 
         
    def preprocess_function(examples):
        current_prefix = prefix if prefix is not None else ""
        inputs = [current_prefix + doc for doc in examples[text]]
        model_inputs = tokenizer(inputs, max_length=inputs_max_length, truncation=True)

        labels = tokenizer(text_target=examples[summary], max_length=labels_max_length, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    
    # Process Hugging Face datasets and remove non-tensor source columns
    tokenized_datasets = raw_datasets.map(
        preprocess_function, 
        batched=True,
        remove_columns=raw_datasets["train"].column_names
    )
    
    # Data collator configured for PyTorch tensors
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, return_tensors="pt")

    # Native PyTorch DataLoaders
    train_dataloader = DataLoader(
        tokenized_datasets["train"], shuffle=True, batch_size=args.batch_size, collate_fn=data_collator
    )
    test_dataloader = DataLoader(
        tokenized_datasets["test"], shuffle=False, batch_size=args.batch_size, collate_fn=data_collator
    )
    validation_dataloader = DataLoader(
        tokenized_datasets["validation"], shuffle=False, batch_size=args.batch_size, collate_fn=data_collator
    )   
    
    # Native PyTorch Optimizer and Linear Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    
    num_update_steps_per_epoch = len(train_dataloader)
    max_train_steps = args.num_train_epochs * num_update_steps_per_epoch
    lr_scheduler = get_scheduler(
        name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=max_train_steps
    )

    # Hugging Face evaluation engine (Replaces Keras Hub)
    metric = evaluate.load("rouge")

    def evaluate_model(dataloader):
        model.eval()
        for batch in tqdm(dataloader, desc="Evaluating"):
            # Move batch tensors to computing hardware
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad():
                generated_tokens = model.generate(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    max_length=labels_max_length,
                )
            
            labels = batch["labels"].cpu().numpy()
            generated_tokens = generated_tokens.cpu().numpy()
            
            # Replace padding tokens introduced by collator back into mask indicators
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            
            decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
            
            metric.add_batch(predictions=decoded_preds, references=decoded_labels)
        
        return metric.compute()

    # Native PyTorch Training Loop
    print("Starting training...")
    progress_bar = tqdm(range(max_train_steps), desc="Training")
    
    for epoch in range(args.num_train_epochs):
        model.train()
        total_loss = 0
        for step, batch in enumerate(train_dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)
            
        avg_train_loss = total_loss / len(train_dataloader)
        print(f"\nEpoch {epoch+1} ended. Average Train Loss: {avg_train_loss:.4f}")
        
        # Validation Evaluation Step
        eval_results = evaluate_model(validation_dataloader)
        print(f"Validation ROUGE-L Score: {eval_results['rougeL']:.4f}")

    # Final Test Set Evaluation Step
    print("\nRunning test evaluation...")
    test_results = evaluate_model(test_dataloader)
    print(f"Test Set ROUGE-L Score: {test_results['rougeL']:.4f}")

    # Save finalized model configuration and weights
    print(f"Saving final model configuration artifacts to {args.output_dir}")
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

if __name__ == "__main__":
    main()
```

### Hyperparameters (for the training script and custom AI job)

The first step to training our model other than setting up our datasets is to set our **hyperparameters**. Hyperparameters depend on your training script and for this one we need to identify our model, the location of our train and test files, etc. 

The batch_size, inputs_max_length, num_train_epochs, and labels_max_length already have defualts setting same as the ones we used in the first section of this tutorial!

In [ ]:
#to view options and defaults you can run the command below
!python autopkg-summarizer/trainer/task.py --help

In [ ]:
#Parameters for task.py script
CHECKPOINT = "google/flan-t5-small"
train_file=training_input_path
test_file=test_input_path
validation_file=validation_input_path
text_column="article"
summary_column="abstract"
source_prefix="summarize: " 
output_dir= f'gs://{BUCKET}'

For custom AI we need to set the machine type, the accelerator for GPUs, and prebuilt docker image that will run our training. See here for more available containers: https://cloud.google.com/vertex-ai/docs/training/pre-built-containers.

In [ ]:
#Parameters for custom AI job
display_name='flan-t5-training-tf'
BASE_GPU_IMAGE_tf='us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-12.py310:latest'
machine_type='n1-standard-4'
accelerator_type='NVIDIA_TESLA_V100'

### Submit Custom AI Training Job

Finally we can submit our training via a custom job! It will first deploy the container that we specified and then submit our model for training. This custom job can take 15 - 20 min using our sample datasets.

In [ ]:
!gcloud ai custom-jobs create \
--region=us-central1 \
--display-name=$display_name \
--args=--model_name_or_path=$CHECKPOINT \
--args=--train_file=$train_file \
--args=--test_file=$test_file \
--args=--validation_file=$validation_file \
--args=--text_column=$text_column \
--args=--summary_column=$summary_column \
--args=--output_dir=gs://$BUCKET \
--args=--source_prefix=$source_prefix \
--worker-pool-spec=machine-type=$machine_type,replica-count=1,accelerator-type=$accelerator_type,executor-image-uri=$BASE_GPU_IMAGE_tf,local-package-path=autopkg-summarizer,python-module=trainer.task

Once you start training the output from the command line should show you the command to use to view the progress of your training via the command `gcloud ai custom-jobs stream-logs <`. You can also monitor and view logs on the console by going to `Vertex AI > Training > Custom Jobs`
select your custom job and click on "View Logs"

### Deploy the Model

### Upload the Model to Vertex AI's Model Registry

Once our model is done training you should see a model_save.pd file in your bucket. We will need this inorder to upload our model to the Model Registry. Here we are specifiying a prebuilt docker image that will run our predictions, the name of our model and the directory in our bucket that holds our **model_save.pd** file.

In [ ]:
TF_PREDICTION_IMAGE_URI_RUNTIME = 'us-docker.pkg.dev/vertex-ai-restricted/prediction/tf_opt-gpu.2-12:latest'

In [ ]:
from google.cloud import aiplatform as vertexai
from google.cloud import aiplatform

#give your model a name
MODEL_DISPLAY_NAME = "summarizer-tf-runtime"
MODEL_DESCRIPTION = "summarizes scientific texts and pdfs" #optional

#add your project ID and location
project='<PROJECT_ID>'
location='<LOCATION>'

vertexai.init(project=project, location=location, staging_bucket=BUCKET)


model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    description=MODEL_DESCRIPTION,
    serving_container_image_uri=TF_PREDICTION_IMAGE_URI_RUNTIME,
    serving_container_args=["--allow_precompilation", "--allow_compression", "--use_tfrt"],
    artifact_uri=f'gs://{BUCKET}/saved_model_artifacts_tf', #directory where our artifacts are in our bucket
)

### Create a Endpoint and Deploy it to our Model

A **endpoint** is how the user of the model can communicate with the model. A single model endpoint responds by returning a single inference from at least one model. It can take 20 min or more to establish a endpoint.

In [ ]:
ENDPOINT_DISPLAY_NAME = "summarizer-endpoint" 
endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_DISPLAY_NAME)

model_endpoint = model.deploy(
    endpoint=endpoint,
    deployed_model_display_name=MODEL_DISPLAY_NAME,
    machine_type="n1-standard-8",
    accelerator_type="NVIDIA_TESLA_V100",
    accelerator_count=1,
    traffic_percentage=100,
    deploy_request_timeout=1200,
    sync=True,
)

Here we are creating a endpoint and deploying our model to said endpoint. We are deploying our endpoint using 1 GPU which can take 20min to run, feel free to try out other machine types that utilize more GPUs.

## Conclusion
Here you learned how to deploy and fine tune a Hugging Face model on Google Cloud

## Clean Up

**Warning:** Once you are done don't forget to delete your endpoint, model, buckets, and shutdown or delete your Vertex AI notebook to avoid additional charges!

First we will delete our custom job. The command below will list custom jobs allowing you to gather the job id from the field called **'name:projects/<PROJECT_ID>/locations/us-central1/customJobs/<JOB_ID>'**

In [ ]:
!gcloud ai custom-jobs list --project=$project --region=$location

In [ ]:
from google.cloud import aiplatform
custom_job_id='<Custom_Job_ID_from_List>'

def delete_custom_job_sample(custom_job_id: str,
    project: str = project,
    location: str = location,
    api_endpoint: str = f'{location}-aiplatform.googleapis.com',
    timeout: int = 300,
):
    # The AI Platform services require regional API endpoints.
    client_options = {"api_endpoint": api_endpoint}
    # Initialize client that will be used to create and send requests.
    # This client only needs to be created once, and can be reused for multiple requests.
    client = aiplatform.gapic.JobServiceClient(client_options=client_options)
    name = client.custom_job_path(
        project=project, location=location, custom_job=custom_job_id
    )
    response = client.delete_custom_job(name=name)
    print("Long running operation:", response.operation.name)
    delete_custom_job_response = response.result(timeout=timeout)
    print("delete_custom_job_response:", delete_custom_job_response)
    
delete_custom_job_sample(custom_job_id)

Now we will undeploy our model, delete endpoints, and delete finally our model!

In [ ]:
model_endpoint.undeploy_all()
model_endpoint.delete()
model.delete()

Delete custom container stored in Custom Registry or Artifacr Registry. List the images to gather the tag id.

In [ ]:
#list the containers
!gcloud container images list-tags gcr.io/$project/cloudai-autogenerated/$display_name

In [ ]:
#Save the tag ID
tag_id='<TAG_ID>'

In [ ]:
#delete 
!gcloud container images delete gcr.io/$project/cloudai-autogenerated/$display_name:$tag_id --force-delete-tags --quiet

And finally delete our bucket

In [ ]:
!gcloud storage rm --recursive gs://$BUCKET/